# Compare EDM4hep versions and update reading functions

In [4]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("../")
from utils import load_root_file
from edm4hep_utils import build_calo_df, build_particle_df, build_tracker_df, load_edm4hep_file

from edm4hep_utils import pixel_readouts, strip_readouts
all_tracker_readouts = pixel_readouts + strip_readouts

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Roadmap

1. Load in reglular low threshold edm4hep file
2. Get histogram binned by vz, vr
3. Load in tv handled edm4hep file


## Loading

In [5]:
base_dir = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/single_particle_pion_10GeV/"
event_num = 0

In [6]:
old_edm4hep_file = f"{base_dir}/v1/single_run_test_low_threshold/edm4hep.root"
old_root_event = uproot.open(old_edm4hep_file)["events"]

In [14]:
new_edm4hep_file = f"{base_dir}/v2/single_run_test_low_threshold/edm4hep.root"
new_root_event = uproot.open(new_edm4hep_file)["events"]

In [15]:
old_root_event.keys()

['ECalBarrelCollection',
 'ECalBarrelCollection/ECalBarrelCollection.cellID',
 'ECalBarrelCollection/ECalBarrelCollection.energy',
 'ECalBarrelCollection/ECalBarrelCollection.position.x',
 'ECalBarrelCollection/ECalBarrelCollection.position.y',
 'ECalBarrelCollection/ECalBarrelCollection.position.z',
 'ECalBarrelCollection/ECalBarrelCollection.contributions_begin',
 'ECalBarrelCollection/ECalBarrelCollection.contributions_end',
 '_ECalBarrelCollection_contributions',
 '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.index',
 '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.collectionID',
 'ECalBarrelCollectionContributions',
 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.PDG',
 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.energy',
 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.time',
 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.stepPosition.x',


In [16]:
new_root_event.keys()

['ECalBarrelCollection',
 'ECalBarrelCollection/ECalBarrelCollection.cellID',
 'ECalBarrelCollection/ECalBarrelCollection.energy',
 'ECalBarrelCollection/ECalBarrelCollection.position.x',
 'ECalBarrelCollection/ECalBarrelCollection.position.y',
 'ECalBarrelCollection/ECalBarrelCollection.position.z',
 'ECalBarrelCollection/ECalBarrelCollection.contributions_begin',
 'ECalBarrelCollection/ECalBarrelCollection.contributions_end',
 '_ECalBarrelCollection_contributions',
 '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.index',
 '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.collectionID',
 'ECalBarrelCollectionContributions',
 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.PDG',
 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.energy',
 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.time',
 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.stepPosition.x',


Seems to only be the MCParticle vs. particle issue

## Reading Functions

In [17]:
event = load_edm4hep_file(old_edm4hep_file, event_num=event_num)

In [18]:
event = load_edm4hep_file(new_edm4hep_file, event_num=event_num)

In [19]:
tracker_df = event["tracker_df"]
tracker_df.columns

Index(['cellID', 'time', 'pathLength', 'quality', 'x', 'y', 'z', 'px', 'py',
       'pz', 'EDep', 'particle_id', 'r', 'R', 'phi', 'theta', 'eta', 'pt',
       'detector'],
      dtype='object')

In [20]:
parents_df = event["parents_df"]

daughters_df = event["daughters_df"]

particles_df = event["particles_df"]

# Create a column from the index
particles_df.columns

Index(['PDG', 'generatorStatus', 'simulatorStatus', 'charge', 'time', 'mass',
       'vx', 'vy', 'vz', 'px', 'py', 'pz', 'endpoint_x', 'endpoint_y',
       'endpoint_z', 'parents_begin', 'parents_end', 'daughters_begin',
       'daughters_end', 'pt', 'p', 'eta', 'phi'],
      dtype='object')

In [21]:
hits_df = event["calo_hits_df"]
hits_df.columns

Index(['cellID', 'energy', 'x', 'y', 'z', 'contribution_begin',
       'contribution_end', 'r', 'R', 'phi', 'theta', 'eta', 'detector'],
      dtype='object')

In [22]:
hits_df

,cellID,energy,x,y,z,contribution_begin,contribution_end,r,R,phi,theta,eta,detector
0,55731877893429264,0.000346,663.032349,1080.951050,1004.700012,0,1,1268.095825,1617.865601,1.020598,0.900774,0.726637,ECalBarrelCollection
1,55731877893462032,0.000236,664.964905,1085.616577,1004.700012,1,2,1273.083618,1621.778076,1.021220,0.902684,0.724202,ECalBarrelCollection
2,56013348575238160,0.000193,671.609253,1088.330444,1009.799988,2,3,1278.875366,1629.484009,1.017898,0.902429,0.724527,ECalBarrelCollection
3,56294823551981584,0.000243,673.541809,1092.996094,1014.900024,3,4,1283.860962,1636.557617,1.018525,0.901871,0.725238,ECalBarrelCollection
4,56576298528725008,0.000189,675.474365,1097.661621,1020.000000,4,5,1288.847046,1643.632080,1.019148,0.901318,0.725943,ECalBarrelCollection
...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,12666335303718931,0.000007,938.468994,1560.120605,1320.000000,1703,1704,1820.631836,2248.799561,1.029248,0.943473,0.673046,HCalBarrelCollection
340,14073705892370451,0.001052,1005.219116,1642.875732,1470.000000,1704,1706,1926.007812,2422.892090,1.021698,0.918877,0.703703,HCalBarrelCollection
341,12947810280429587,0.001027,938.468994,1560.120605,1350.000000,1706,1714,1820.631836,2266.539307,1.029248,0.932757,0.686334,HCalBarrelCollection
342,12947801690494995,0.000028,993.901794,1537.159546,1350.000000,1714,1715,1830.491821,2274.467041,0.996827,0.935339,0.683123,HCalBarrelCollection


In [23]:
contrib_df = event["calo_contrib_df"]
contrib_df.columns


Index(['PDG', 'energy', 'time', 'step_x', 'step_y', 'step_z', 'particle_id',
       'detector'],
      dtype='object')